# table_c_j

## communal and daily grid in the PostegreSQL database

**Role:** Initializes the daily spatio-temporal grid for municipalities and creates materialized views for spatial neighbors to compute future geographic contagion. The scope is strictly focused on the PACA region from 2016 to 2025.

**Inputs:**
- PostgreSQL tables: `incendies.commune`, `incendies.incendie`[cite: 9, 10]

**Outputs:**
- PostgreSQL table: `incendies.commune_jour` (base daily grid initialized with the `has_fire` target)[cite: 9, 10]
- PostgreSQL materialized views: `incendies.mv_communes_voisines_10km`, `incendies.mv_communes_voisines_20km`, `incendies.mv_communes_voisines_50km`[cite: 9, 10]

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import sqlalchemy
import psycopg2
import pandas as pd

sys.path.append(str(Path.cwd().parent))

load_dotenv()

from src.config import (
    HOST,
    NAME,
    USER,
    PASSWORD,
    PORT,
    URI,
)
from src.db import read_query



engine = create_engine(
    URI,
    connect_args={"options": "-csearch_path=incendies_schema,public"}
)

# Test de connexion
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT version()"))
        print("Connexion réussie !")
        print("Version PostgreSQL :", result.scalar())

        dbs = conn.execute(
            text("SELECT datname FROM pg_database WHERE datistemplate = false ORDER BY datname")
        ).fetchall()
        print("Bases de données :", [db[0] for db in dbs])
except Exception as e:
    print("Erreur de connexion :", e)


[2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Connexion réussie !
Version PostgreSQL : PostgreSQL 17.5 (Debian 17.5-1.pgdg110+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 10.2.1-6) 10.2.1 20210110, 64-bit
Bases de données : ['incendies', 'postgres']


In [2]:
# DATABASE_NAME="incendies"

def get_engine():
    return create_engine(
        URI,
        connect_args={"options": "-csearch_path=incendies,public"},
    )

engine = get_engine()

## `commune_jour` table update (after import)

During the docker compose build, the incendies schema and the  `commune_jour` table were created<br>
but no data was imported

Data is imported now

filtered on :
- 10 years (2016 -2025 included)
- PACA region (south east of France)


In [ ]:
sql = text("""
    INSERT INTO commune_jour (id_commune, date_jour, has_fire)
    SELECT
        c.id_commune,
        d.jour::date,
        0
    FROM
        commune c
    CROSS JOIN
        generate_series(
            '2016-01-01'::date,
            '2025-12-31'::date,
            '1 day'::interval
        ) AS d(jour)
    WHERE
        c.region = 17;
""")

with engine.begin() as conn:
    conn.execute(sql)

In [10]:
sql = text("""
    UPDATE commune_jour cj
    SET has_fire = 1
    FROM incendie i
    JOIN commune c ON c.code_insee = i.code_insee
    WHERE cj.id_commune = c.id_commune
      AND cj.date_jour = i.date_premiere_alerte::date
      AND c.region = 17;
""")

with engine.begin() as conn:
    conn.execute(sql)

### Feature engineering

In [11]:
# récupération des dates du jour
dates = pd.read_sql(
    "SELECT DISTINCT date_jour FROM commune_jour ORDER BY date_jour",
    engine,
)

#### `nb_incendies_30j`, `nb_incendies_90j`, `nb_incendies_365j` update

In [12]:
ddl = [
    """
    CREATE INDEX IF NOT EXISTS idx_incendie_date_insee
        ON incendie (date_premiere_alerte, code_insee);

    """
]

with engine.connect() as conn:
    conn.execute(text("SET search_path TO incendies, public"))
    for stmt in ddl:
        conn.execute(text(stmt))
        conn.commit()

#### `v_commune_paca` vue focused on the PACA cities

In [14]:
ddl = [
    """
    CREATE VIEW v_commune_paca AS
    SELECT
        *
    FROM commune
    WHERE region = 17;
    """
]
with engine.connect() as conn:
    conn.execute(text("SET search_path TO incendies, public"))
    for stmt in ddl:
        conn.execute(text(stmt))
        conn.commit()

In [ ]:
sql = text("""
    WITH calcul_incendies AS (
        SELECT
            c.id_commune,
            COUNT(*) FILTER (
                WHERE i.date_premiere_alerte::date >= :date_jour - INTERVAL '30 days'
            ) AS nb_30j,
            COUNT(*) FILTER (
                WHERE i.date_premiere_alerte::date >= :date_jour - INTERVAL '90 days'
            ) AS nb_90j,
            COUNT(*) AS nb_365j
        FROM v_commune_paca c
        JOIN incendie i ON i.code_insee = c.code_insee
            AND i.date_premiere_alerte::date >= :date_jour - INTERVAL '365 days'
            AND i.date_premiere_alerte::date < :date_jour
        GROUP BY c.id_commune
    )
    UPDATE commune_jour cj
    SET
        nb_incendies_30j = COALESCE(calc.nb_30j, 0),
        nb_incendies_90j = COALESCE(calc.nb_90j, 0),
        nb_incendies_365j = COALESCE(calc.nb_365j, 0)
    FROM v_commune_paca c
    LEFT JOIN calcul_incendies calc ON calc.id_commune = c.id_commune
    WHERE cj.id_commune = c.id_commune
      AND cj.date_jour = :date_jour;
""")

with engine.begin() as conn:
    for jour in dates["date_jour"]:
        conn.execute(sql, {"date_jour": jour})

#### `surface_totale_5a` update

In [ ]:
sql = text("""
WITH date_bornes AS (
    SELECT
        CAST(:date_jour AS DATE) AS d_end,
        CAST(:date_jour AS DATE) - INTERVAL '1825 days' AS d_5a
),
calcul_surface AS (
    SELECT
        c.id_commune,
        SUM(i.surface_parcourue) AS surface_totale_5a
    FROM date_bornes b
    CROSS JOIN incendie i
    JOIN v_commune_paca c ON i.code_insee = c.code_insee
    WHERE i.date_premiere_alerte >= b.d_5a
      AND i.date_premiere_alerte < b.d_end
    GROUP BY c.id_commune
)
UPDATE commune_jour cj
SET surface_totale_5a = COALESCE(calc.surface_totale_5a, 0)
FROM v_commune_paca c
LEFT JOIN calcul_surface calc ON calc.id_commune = c.id_commune
WHERE cj.id_commune = c.id_commune
  AND cj.date_jour = :date_jour;
""")

with engine.begin() as conn:
    for jour in dates["date_jour"]:
        conn.execute(sql, {"date_jour": jour})

#### `buffer_10km`, `buffer_20km`, `buffer_50km` update


In [ ]:
# sql = text("""
#     WITH communes AS (
#         SELECT
#             c1.id_commune AS ref_commune,
#             c2.code_insee AS code_insee
#         FROM v_commune_paca c1
#         JOIN localisation l1 ON l1.id_localisation = c1.localisation
#         JOIN localisation l2 ON ST_DWithin(
#             ST_Transform(l1.geom, 2154),
#             ST_Transform(l2.geom, 2154),
#             10000
#         )
#         JOIN commune c2 ON c2.localisation = l2.id_localisation
#         WHERE
#             c2.id_commune != c1.id_commune
#     ),
#     calcul_incendies AS (
#     SELECT
#         v.ref_commune,
#         COUNT(i.id_incendie) AS feux_du_jour
#     FROM communes v
#     LEFT JOIN incendie i ON i.code_insee = v.code_insee
#             AND CAST(i.date_premiere_alerte AS DATE) = :date_jour
#             AND CAST(i.date_premiere_alerte AS DATE) BETWEEN :date_jour AND (:date_jour + INTERVAL '30 days')
#     GROUP BY v.ref_commune
#     )
#     UPDATE commune_jour cj
#     SET buffer_10km = calc.feux_du_jour
#     FROM calcul_incendies calc
#     WHERE cj.id_commune = calc.ref_commune
#     AND cj.date_jour = :date_jour;
# """)

# with engine.connect() as conn:
#     for jour in dates["date_jour"]:
#         conn.execute(sql, {"date_jour": jour})
#         conn.commit()



In [18]:
# Création de vueq matérialisées
ddl = [
    """
    CREATE MATERIALIZED VIEW mv_communes_voisines_10km AS
    SELECT
        c1.id_commune AS ref_commune,
        c2.code_insee AS code_insee
    FROM v_commune_paca c1
    JOIN localisation l1 ON l1.id_localisation = c1.localisation
    JOIN localisation l2 ON ST_DWithin(
        ST_Transform(l1.geom, 2154),
        ST_Transform(l2.geom, 2154),
        10000
    )
    JOIN commune c2 ON c2.localisation = l2.id_localisation
    WHERE c2.id_commune != c1.id_commune;
    """,
    """
    CREATE MATERIALIZED VIEW mv_communes_voisines_20km AS
    SELECT
        c1.id_commune AS ref_commune,
        c2.code_insee AS code_insee
    FROM v_commune_paca c1
    JOIN localisation l1 ON l1.id_localisation = c1.localisation
    JOIN localisation l2 ON ST_DWithin(
        ST_Transform(l1.geom, 2154),
        ST_Transform(l2.geom, 2154),
        20000
    )
    JOIN commune c2 ON c2.localisation = l2.id_localisation
    WHERE c2.id_commune != c1.id_commune;
    """,
    """
    CREATE MATERIALIZED VIEW mv_communes_voisines_50km AS
    SELECT
        c1.id_commune AS ref_commune,
        c2.code_insee AS code_insee
    FROM v_commune_paca c1
    JOIN localisation l1 ON l1.id_localisation = c1.localisation
    JOIN localisation l2 ON ST_DWithin(
        ST_Transform(l1.geom, 2154),
        ST_Transform(l2.geom, 2154),
        50000
    )
    JOIN commune c2 ON c2.localisation = l2.id_localisation
    WHERE c2.id_commune != c1.id_commune;
    """
]

with engine.connect() as conn:
    conn.execute(text("SET search_path TO incendies, public"))
    for stmt in ddl:
        conn.execute(text(stmt))
        conn.commit()

### Create indexes
to speed up the compute

In [19]:
ddl = [
    """
    CREATE INDEX IF NOT EXISTS idx_mv_cv_ref_commune_10
        ON mv_communes_voisines_10km(ref_commune);
    """,
    """
    CREATE INDEX IF NOT EXISTS idx_mv_cv_code_insee_10
        ON mv_communes_voisines_10km(code_insee);
    """,
    """
    CREATE INDEX IF NOT EXISTS idx_mv_cv_ref_commune_20
        ON mv_communes_voisines_20km(ref_commune);
    """,
    """
    CREATE INDEX IF NOT EXISTS idx_mv_cv_code_insee_20
        ON mv_communes_voisines_20km(code_insee);
    """,
    """
    CREATE INDEX IF NOT EXISTS idx_mv_cv_ref_commune_50
        ON mv_communes_voisines_50km(ref_commune);
    """,
    """
    CREATE INDEX IF NOT EXISTS idx_mv_cv_code_insee_50
        ON mv_communes_voisines_50km(code_insee);
    """,
]

with engine.connect() as conn:
    conn.execute(text("SET search_path TO incendies, public"))
    for stmt in ddl:
        conn.execute(text(stmt))
        conn.commit()
sql = text("""
    WITH calcul_incendies AS (
        SELECT
            v.ref_commune,
            COUNT(i.id_incendie) AS feux_du_jour
        FROM mv_communes_voisines_10km v
        LEFT JOIN incendie i ON i.code_insee = v.code_insee
            AND i.date_premiere_alerte >= :date_jour
            AND i.date_premiere_alerte < (:date_jour + INTERVAL '30 day')
        GROUP BY v.ref_commune
    )
    UPDATE commune_jour cj
    SET buffer_10km = calc.feux_du_jour
    FROM calcul_incendies calc
    WHERE cj.id_commune = calc.ref_commune
    AND cj.date_jour = :date_jour;
""")

with engine.connect() as conn:
    for jour in dates["date_jour"]:
        conn.execute(sql, {"date_jour": jour})
        conn.commit()


In [ ]:
sql = text("""
    WITH calcul_incendies AS (
        SELECT
            v.ref_commune,
            COUNT(i.id_incendie) AS feux_du_jour
        FROM mv_communes_voisines_10km v
        LEFT JOIN incendie i ON i.code_insee = v.code_insee
            AND i.date_premiere_alerte::date >= :date_jour - INTERVAL '30 days'
            AND i.date_premiere_alerte::date < :date_jour
        GROUP BY v.ref_commune
    )
    UPDATE commune_jour cj
    SET buffer_10km = COALESCE(calc.feux_du_jour, 0)
    FROM calcul_incendies calc
    WHERE cj.id_commune = calc.ref_commune
      AND cj.date_jour = :date_jour;
""")

with engine.begin() as conn:
    for jour in dates["date_jour"]:
        conn.execute(sql, {"date_jour": jour})

In [ ]:
sql = text("""
    WITH calcul_incendies AS (
        SELECT
            v.ref_commune,
            COUNT(i.id_incendie) AS feux_du_jour
        FROM mv_communes_voisines_20km v
        LEFT JOIN incendie i ON i.code_insee = v.code_insee
            AND i.date_premiere_alerte::date >= :date_jour - INTERVAL '30 days'
            AND i.date_premiere_alerte::date < :date_jour
        GROUP BY v.ref_commune
    )
    UPDATE commune_jour cj
    SET buffer_20km = COALESCE(calc.feux_du_jour, 0)
    FROM calcul_incendies calc
    WHERE cj.id_commune = calc.ref_commune
      AND cj.date_jour = :date_jour;
""")

with engine.begin() as conn:
    for jour in dates["date_jour"]:
        conn.execute(sql, {"date_jour": jour})

In [ ]:
sql = text("""
    WITH calcul_incendies AS (
        SELECT
            v.ref_commune,
            COUNT(i.id_incendie) AS feux_du_jour
        FROM mv_communes_voisines_50km v
        LEFT JOIN incendie i ON i.code_insee = v.code_insee
            AND i.date_premiere_alerte::date >= :date_jour - INTERVAL '30 days'
            AND i.date_premiere_alerte::date < :date_jour
        GROUP BY v.ref_commune
    )
    UPDATE commune_jour cj
    SET buffer_50km = COALESCE(calc.feux_du_jour, 0)
    FROM calcul_incendies calc
    WHERE cj.id_commune = calc.ref_commune
      AND cj.date_jour = :date_jour;
""")

with engine.begin() as conn:
    for jour in dates["date_jour"]:
        conn.execute(sql, {"date_jour": jour})